# 01 — Representatividade da Amostra (PROMISE NFR)

Verifica se a amostra `n=100, seed=42` mantém a distribuição de categorias
proporcional ao dataset completo (`n=625`).

**Checklist:**
- [ ] Distribuição FR vs NFR (proporção e χ²)
- [ ] Distribuição das 11 subcategorias PROMISE
- [ ] Alertas para categorias raras (n < 10 no dataset completo)
- [ ] Conclusão: amostra é representativa para uso nos experimentos?

In [ ]:
import sys
from pathlib import Path

# Garante que o root do projeto está no path
ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from scipy.stats import chi2_contingency

CSV_PATH = ROOT / "datasets/data/promise_nfr/promise_nfr_pt.csv"

CATEGORY_LABELS = {
    "F":  "Functional (FR)",
    "A":  "Availability",
    "FT": "Fault Tolerance",
    "LF": "Look & Feel",
    "MN": "Maintainability",
    "O":  "Operational",
    "PE": "Performance",
    "PO": "Portability",
    "SC": "Scalability",
    "SE": "Security",
    "US": "Usability",
}

SAMPLE_N    = 100
SAMPLE_SEED = 42

df_full   = pd.read_csv(CSV_PATH)
df_full["class"] = df_full["class"].str.strip().str.upper()
df_sample = df_full.sample(n=SAMPLE_N, random_state=SAMPLE_SEED)

print(f"Dataset completo : {len(df_full)} requisitos")
print(f"Amostra          : {len(df_sample)} requisitos  (n={SAMPLE_N}, seed={SAMPLE_SEED})")

## 1. FR vs NFR

In [ ]:
def fr_nfr(df):
    return df["class"].map(lambda c: "FR" if c == "F" else "NFR")

full_type   = fr_nfr(df_full)
sample_type = fr_nfr(df_sample)

# Tabela comparativa
tbl = pd.DataFrame({
    "Full n":     full_type.value_counts(),
    "Sample n":   sample_type.value_counts(),
}).fillna(0).astype(int)
tbl["Full %"]   = (tbl["Full n"]   / tbl["Full n"].sum()   * 100).round(1)
tbl["Sample %"] = (tbl["Sample n"] / tbl["Sample n"].sum() * 100).round(1)
tbl["Δ %"]      = (tbl["Sample %"] - tbl["Full %"]).round(1)

display(tbl)

# Teste χ²
categories = sorted(tbl.index)
contingency = [[tbl.loc[c, "Full n"] for c in categories],
               [tbl.loc[c, "Sample n"] for c in categories]]
chi2, p, *_ = chi2_contingency(contingency)
verdict = "✓ Representativa (p ≥ 0.05)" if p >= 0.05 else "✗ Divergente (p < 0.05)"
print(f"\nχ² = {chi2:.4f}   p = {p:.4f}   → {verdict}")

# Gráfico
fig, ax = plt.subplots(figsize=(5, 3.5))
x = np.arange(len(categories))
width = 0.35
ax.bar(x - width/2, [tbl.loc[c, "Full %"] for c in categories],   width, label="Full (n=625)", color="#4C72B0")
ax.bar(x + width/2, [tbl.loc[c, "Sample %"] for c in categories], width, label=f"Sample (n={SAMPLE_N})", color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(categories)
ax.set_ylabel("% dos requisitos")
ax.set_title("Distribuição FR vs NFR")
ax.legend(); ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.savefig("fr_nfr_distribution.png", dpi=150)
plt.show()

## 2. Subcategorias PROMISE (11 classes)

In [ ]:
full_cat   = df_full["class"].map(CATEGORY_LABELS)
sample_cat = df_sample["class"].map(CATEGORY_LABELS)

tbl2 = pd.DataFrame({
    "Full n":   full_cat.value_counts(),
    "Sample n": sample_cat.value_counts(),
}).fillna(0).astype(int)
tbl2["Full %"]   = (tbl2["Full n"]   / tbl2["Full n"].sum()   * 100).round(1)
tbl2["Sample %"] = (tbl2["Sample n"] / tbl2["Sample n"].sum() * 100).round(1)
tbl2["Δ %"]      = (tbl2["Sample %"] - tbl2["Full %"]).round(1)
tbl2 = tbl2.sort_values("Full n", ascending=False)

display(tbl2)

cats = tbl2.index.tolist()
chi2, p, *_ = chi2_contingency([tbl2["Full n"].tolist(), tbl2["Sample n"].tolist()])
verdict = "✓ Representativa (p ≥ 0.05)" if p >= 0.05 else "✗ Divergente (p < 0.05)"
print(f"\nχ² = {chi2:.4f}   p = {p:.4f}   → {verdict}")

# Gráfico horizontal
fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(cats))
height = 0.35
ax.barh(y + height/2, tbl2["Full %"],   height, label="Full (n=625)", color="#4C72B0")
ax.barh(y - height/2, tbl2["Sample %"], height, label=f"Sample (n={SAMPLE_N})", color="#DD8452")
ax.set_yticks(y); ax.set_yticklabels(cats, fontsize=9)
ax.set_xlabel("% dos requisitos")
ax.set_title("Distribuição das Subcategorias PROMISE")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
plt.tight_layout()
plt.savefig("subcategory_distribution.png", dpi=150)
plt.show()

## 3. Categorias raras e recomendação

In [ ]:
rare = df_full["class"].value_counts()[df_full["class"].value_counts() < 10]
if rare.empty:
    print("Nenhuma categoria com menos de 10 exemplos no dataset completo.")
else:
    print("⚠  Categorias raras no dataset completo (n < 10):")
    for cat, cnt in rare.items():
        label = CATEGORY_LABELS.get(cat, cat)
        in_sample = (df_sample["class"] == cat).sum()
        print(f"   {label:<20} → full={cnt}  sample={in_sample}")

print("\n--- Conclusão ---")
print("Amostra aleatória simples (seed=42) é adequada para experimentos preliminares.")
print("Para resultados finais do SBCARS, recomenda-se amostragem estratificada")
print("para garantir representação mínima das categorias raras (PO, FT).")